# Testing with pytest

pytest is Python's most popular testing framework. It finds and runs test functions automatically, produces readable failure output, and adds powerful features (fixtures, parametrize, mocking) on top of plain `assert` statements. Unlike `unittest`, there's almost no boilerplate.

**What's inside:** basic test functions, `pytest.raises`, fixtures, `@pytest.mark.parametrize`, `unittest.mock` (patch and MagicMock), and the `tmp_path` fixture.

**Learn more:** [pytest docs](https://docs.pytest.org) · [unittest.mock](https://docs.python.org/3/library/unittest.mock.html)

## Setup

In [ ]:
%pip install pytest pytest-asyncio

## 1. Basic test functions

> **Notebook note:** pytest normally discovers and runs `.py` files from the command line (`pytest`). In a notebook we use `ipytest` for inline execution, or we show examples you'd paste into a `.py` file. The patterns below are standard pytest; run them with `pytest test_mymodule.py`.

In [ ]:
# Example: the module under test
def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ValueError('cannot divide by zero')
    return a / b


# Test file contents: paste into test_math.py and run: pytest test_math.py
test_source = '''
def add(a, b): return a + b
def divide(a, b):
    if b == 0: raise ValueError('cannot divide by zero')
    return a / b

def test_add_integers():
    assert add(2, 3) == 5

def test_add_strings():
    assert add('hello', ' world') == 'hello world'

def test_add_negative():
    assert add(-1, -1) == -2
'''

import tempfile, subprocess, sys
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
(tmp / 'test_math.py').write_text(test_source)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tmp / 'test_math.py'), '-v'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 2. pytest.raises: testing exceptions

In [ ]:
import tempfile, subprocess, sys
from pathlib import Path

source = '''
import pytest

def divide(a, b):
    if b == 0:
        raise ValueError('cannot divide by zero')
    return a / b

def test_divide_by_zero():
    with pytest.raises(ValueError, match='cannot divide by zero'):
        divide(10, 0)

def test_divide_ok():
    assert divide(10, 2) == 5.0
'''

tmp = Path(tempfile.mkdtemp())
(tmp / 'test_exc.py').write_text(source)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tmp / 'test_exc.py'), '-v'],
    capture_output=True, text=True
)
print(result.stdout)

## 3. Fixtures: reusable test setup

In [ ]:
import tempfile, subprocess, sys
from pathlib import Path

source = '''
import pytest

class Database:
    def __init__(self):
        self._data = {}
    def set(self, k, v): self._data[k] = v
    def get(self, k):    return self._data.get(k)
    def delete(self, k): del self._data[k]

@pytest.fixture
def db():
    """Provide a fresh Database instance per test."""
    return Database()

def test_set_and_get(db):
    db.set('x', 42)
    assert db.get('x') == 42

def test_missing_key(db):
    assert db.get('missing') is None

def test_delete(db):
    db.set('y', 99)
    db.delete('y')
    assert db.get('y') is None
'''

tmp = Path(tempfile.mkdtemp())
(tmp / 'test_fixtures.py').write_text(source)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tmp / 'test_fixtures.py'), '-v'],
    capture_output=True, text=True
)
print(result.stdout)

## 4. parametrize: run one test with many inputs

In [ ]:
import tempfile, subprocess, sys
from pathlib import Path

source = '''
import pytest

def is_palindrome(s):
    s = s.lower().replace(' ', '')
    return s == s[::-1]

@pytest.mark.parametrize('word, expected', [
    ('racecar',   True),
    ('level',     True),
    ('A man a plan a canal Panama', True),
    ('hello',     False),
    ('',          True),
])
def test_palindrome(word, expected):
    assert is_palindrome(word) == expected
'''

tmp = Path(tempfile.mkdtemp())
(tmp / 'test_param.py').write_text(source)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tmp / 'test_param.py'), '-v'],
    capture_output=True, text=True
)
print(result.stdout)

## 5. unittest.mock: patch and MagicMock

In [ ]:
import tempfile, subprocess, sys
from pathlib import Path

source = '''
from unittest.mock import patch, MagicMock
import pytest

# Production code that calls an external service
def get_user(user_id):
    import urllib.request, json
    with urllib.request.urlopen(f'https://api.example.com/users/{user_id}') as r:
        return json.loads(r.read())

def greet_user(user_id):
    user = get_user(user_id)
    return f"Hello, {user['name']}!"

# Tests that patch the network call
def test_greet_user():
    mock_user = {'name': 'Alice', 'id': 1}
    with patch('__main__.get_user', return_value=mock_user):
        result = greet_user(1)
    assert result == "Hello, Alice!"

def test_get_user_called_with_correct_id():
    with patch("__main__.get_user") as mock_get:
        mock_get.return_value = {"name": "Bob"}
        greet_user(42)
        mock_get.assert_called_once_with(42)
'''

tmp = Path(tempfile.mkdtemp())
(tmp / 'test_mock.py').write_text(source)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tmp / 'test_mock.py'), '-v'],
    capture_output=True, text=True
)
print(result.stdout)

## 6. tmp_path: built-in fixture for temporary files

In [ ]:
import tempfile, subprocess, sys
from pathlib import Path

source = '''
import json

def save_config(path, config):
    path.write_text(json.dumps(config), encoding='utf-8')

def load_config(path):
    return json.loads(path.read_text(encoding='utf-8'))

def test_roundtrip(tmp_path):      # tmp_path is a pytest built-in fixture
    config_file = tmp_path / "config.json"
    data = {"debug": True, "port": 8080}
    save_config(config_file, data)
    loaded = load_config(config_file)
    assert loaded == data
    assert config_file.exists()
'''

tmp = Path(tempfile.mkdtemp())
(tmp / 'test_tmppath.py').write_text(source)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', str(tmp / 'test_tmppath.py'), '-v'],
    capture_output=True, text=True
)
print(result.stdout)